# Full-Text Search with Redis

This notebook demonstrates using `RedisFullTextSearch` for BM25-based keyword search.

**Features:**
- BM25 lexical search (keyword matching)
- Field boosting for relevance tuning
- Tag and numeric filtering
- Complementary to vector search

**Use Cases:**
- Exact keyword matching
- When semantic similarity isn't needed
- Hybrid search (combine with vector search)

## Setup

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

REDIS_URL = os.getenv("REDIS_URL", "redis://redis:6379")
print(f"Redis URL: {REDIS_URL}")

Redis URL: redis://redis:6379


## Create Full-Text Search Index

In [2]:
from redis_openai_agents import RedisFullTextSearch

# Create a full-text search index
fts = RedisFullTextSearch(
    name="articles",
    redis_url=REDIS_URL,
)

print(f"Full-text search index created: {fts.name}")

Full-text search index created: articles


## Add Documents

In [3]:
# Sample articles
articles = [
    {
        "title": "Introduction to Redis",
        "content": "Redis is an open-source, in-memory data structure store. It can be used as a database, cache, and message broker.",
        "tags": ["redis", "database", "nosql"],
        "category": "tutorial"
    },
    {
        "title": "Redis Data Structures",
        "content": "Redis supports strings, hashes, lists, sets, and sorted sets. Each data structure has specific use cases.",
        "tags": ["redis", "data-structures"],
        "category": "tutorial"
    },
    {
        "title": "Caching with Redis",
        "content": "Redis is commonly used as a cache layer. It provides fast read and write operations with TTL support.",
        "tags": ["redis", "caching", "performance"],
        "category": "guide"
    },
    {
        "title": "Python Web Development",
        "content": "Python is great for web development. Frameworks like Django and Flask make it easy to build web applications.",
        "tags": ["python", "web", "django", "flask"],
        "category": "tutorial"
    },
    {
        "title": "Redis Clustering",
        "content": "Redis Cluster provides automatic sharding across multiple nodes. It offers high availability and scalability.",
        "tags": ["redis", "clustering", "scalability"],
        "category": "advanced"
    },
]

ids = fts.add_documents(articles)
print(f"Added {len(ids)} articles")

Added 5 articles


## Basic Keyword Search

In [4]:
# Search for articles containing "cache"
results = fts.search(query="cache", k=5)

print(f"Search: 'cache'\n")
for i, r in enumerate(results, 1):
    print(f"{i}. {r['title']}")
    print(f"   Score: {r['score']:.4f}")
    print(f"   Content: {r['content'][:60]}...\n")

Search: 'cache'

1. Caching with Redis
   Score: 0.0000
   Content: Redis is commonly used as a cache layer. It provides fast re...

2. Introduction to Redis
   Score: 0.0000
   Content: Redis is an open-source, in-memory data structure store. It ...



## Multi-word Search

In [5]:
# Search for multiple keywords
results = fts.search(query="redis data structures", k=3)

print(f"Search: 'redis data structures'\n")
for i, r in enumerate(results, 1):
    print(f"{i}. {r['title']} (score: {r['score']:.4f})")

Search: 'redis data structures'

1. Redis Data Structures (score: 0.0000)
2. Introduction to Redis (score: 0.0000)


## Filtered Search

In [6]:
# Search only in tutorials
results = fts.search(
    query="redis",
    k=5,
    filter={"category": "tutorial"}
)

print(f"Search: 'redis' (category=tutorial)\n")
for i, r in enumerate(results, 1):
    print(f"{i}. {r['title']} [{r.get('category', 'N/A')}]")

Search: 'redis' (category=tutorial)

1. Introduction to Redis [tutorial]
2. Redis Data Structures [tutorial]


## Search by Tag

In [7]:
# Search articles with specific tag
results = fts.search(
    query="*",  # Match all
    k=5,
    filter={"tags": "caching"}
)

print(f"Articles tagged 'caching':\n")
for i, r in enumerate(results, 1):
    print(f"{i}. {r['title']}")

Articles tagged 'caching':

1. Caching with Redis


## Document Count

In [8]:
print(f"Total documents: {fts.count()}")

Total documents: 5


## Cleanup

In [9]:
fts.delete_all()
print(f"Deleted all documents. Count: {fts.count()}")

Deleted all documents. Count: 0


## Summary

This notebook demonstrated:

1. **Basic Search** - BM25 keyword matching
2. **Multi-word Search** - Multiple keyword queries
3. **Filtered Search** - Category and tag filtering
4. **Document Management** - Add, count, delete

**When to Use Full-Text vs Vector Search:**
- **Full-Text**: Exact keyword matching, known terminology
- **Vector**: Semantic similarity, paraphrased queries
- **Hybrid**: Combine both for best results